In [ ]:
# for plotting
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import colorcet as cc
import numpy as np
import pickle
import json
from pathlib import Path

sns.set()
sns.set_context('poster')
sns.set_style('ticks')
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = 'cmr10'
plt.rcParams["mathtext.fontset"] = 'cm'
plt.rcParams["axes.formatter.use_mathtext"] = True

In [ ]:
with open('bomex_25m_r20251009/pkl/csd_stats.pkl', 'rb') as f:
    stat_ctl = pickle.load(f)
with open('bomex_25m_ehe18_r20251009/pkl/csd_stats.pkl', 'rb') as f:
    stat_ehe18 = pickle.load(f)
with open('bomex_25m_ehe21_r20251107/pkl/csd_stats.pkl', 'rb') as f:
    stat_ehe21 = pickle.load(f)
with open('bomex_25m_ehe22_r20251107/pkl/csd_stats.pkl', 'rb') as f:
    stat_ehe22 = pickle.load(f)

In [ ]:
nc_ctl, = stat_ctl[0].shape
nc_ehe18, = stat_ehe18[0].shape
nc_ehe21, = stat_ehe21[0].shape
nc_ehe22, = stat_ehe22[0].shape
print(f'{nc_ctl} clouds in CTL simulation')
print(f'{nc_ehe18} clouds in EHE18 simulation')
print(f'{nc_ehe21} clouds in EHE21 simulation')
print(f'{nc_ehe22} clouds in EHE22 simulation')

## Compare the Distributions

In [ ]:
dz = 25 # m
z = np.arange(dz/2, 3000., dz)

In [ ]:
minmf = 0
maxmf = np.max([np.max(stat_ehe18[0]), np.max(stat_ehe21[0]), np.max(stat_ehe22[0]), np.max(stat_ctl[0])]) + 10.0
maxmf = np.log10(maxmf)
print(minmf, maxmf)

In [ ]:
def compare_csd(stat_ehe, stat_ctl, name_ehe, nbins, minmf, maxmf):

    bins = np.linspace(minmf, maxmf, nbins+1)
    fig = plt.figure(figsize=(12,14))
    ax1 = fig.add_axes((0.1, 0.55, 0.85, 0.4))
    ax2 = fig.add_axes((0.1, 0.1, 0.85, 0.4))

    # First panel - histogram comparison
    n, bins_out, patches = ax1.hist([np.log10(stat_ehe[0]), np.log10(stat_ctl[0])], bins=bins, log=True, color=['red', 'black'], label=[name_ehe, 'CTL'], alpha=0.7)
    ax1.axvline(np.median(np.log10(stat_ehe[0])), color='red', linestyle='--')
    ax1.axvline(np.median(np.log10(stat_ctl[0])), color='black', linestyle='--')
    # ax1.legend([name_ehe, 'CTL'])
    ax1.set_ylim(1, 1.0e4)
    # ax1.set_xlabel(f'Mean cloud-base mass flux (kg/s)')
    ax1.set_ylabel('Number of clouds')
    ax1.set_title('Histograms')
    ax1.legend()

    # Second panel - difference plot
    diff = n[0] - n[1]
    bin_centers = (bins[:-1] + bins[1:]) / 2
    ax2.bar(bin_centers, np.where(diff>0.0, diff, np.nan), width=np.diff(bins), color='red', label=rf'{name_ehe} $>$ CTL')
    ax2.bar(bin_centers, np.where(diff>0.0, np.nan, -diff), width=np.diff(bins), color='blue', label=rf'CTL $>$ {name_ehe}')
    ax2.axhline(0, color='black', linestyle='-', linewidth=0.5)
    ax2.set_xlabel(r'$\log_{10}\left<M_b\right>$')
    ax2.set_ylabel(f'Difference ({name_ehe} - CTL)')
    ax2.set_yscale('log')
    ax2.set_ylim(0.1, 1.0e4)
    ax2.legend()

    plt.show()

In [ ]:
nbins = 10 

In [ ]:
compare_csd(stat_ehe18, stat_ctl, 'Smooth all', nbins, minmf, maxmf)

In [ ]:
compare_csd(stat_ehe22, stat_ctl, 'Smooth lower', nbins, minmf, maxmf)

In [ ]:
compare_csd(stat_ehe21, stat_ctl, 'Smooth upper', nbins, minmf, maxmf)